# Merge Drive-stored channel descriptions

This notebook merges the channel-description artifacts produced by the three research notebook subdirectories (`Jules`, `codex`, and `copilot`) into one provenance-preserving JSON object.

For each channel, the merged object contains up to three descriptions keyed by source subdirectory. Each description string is prefixed with its source name, for example `copilot: ...`, so downstream analysis can compare independently generated descriptions without losing provenance.

After writing the merged object, the notebook can encode each source's descriptions and write one symmetric channel-by-channel cosine-similarity matrix per source description set.


## 1) Setup

Mount Google Drive when running in Colab, then define the default Drive locations for each notebook's cached descriptions and for the merged artifact. The notebook can also run outside Colab for dry checks or local testing by editing the paths in the next cell.


In [ ]:
from __future__ import annotations

import csv
import glob
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    print(f'Google Drive was not mounted automatically: {exc}')
    IN_COLAB = False

DRIVE_ROOT = Path('/content/drive/MyDrive')

SOURCE_ARTIFACTS = {
    'Jules': DRIVE_ROOT / 'research/channel_descriptions_and_segments.json',
    'codex': DRIVE_ROOT / 'Graphiko/research/semantic_description_pointcloud_alignment/descriptions/by_channel',
    'copilot': DRIVE_ROOT / 'Research/channel_descriptions/latest/all_channels.json',
}

OUTPUT_PATH = (
    DRIVE_ROOT
    / 'Graphiko/research/merged_channel_descriptions/latest/all_channel_descriptions_by_source.json'
)

SIMILARITY_OUTPUT_DIR = OUTPUT_PATH.parent / 'description_cosine_similarity_matrices'
EMBEDDING_OUTPUT_DIR = OUTPUT_PATH.parent / 'description_embeddings'
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

# Keep source labels out of the encoded text by default so each matrix reflects
# the description content rather than the common `source: ` prefix.
INCLUDE_SOURCE_PREFIX_IN_EMBEDDINGS = False

# Set to False if you only need matrices and do not want per-description vectors on Drive.
WRITE_DESCRIPTION_EMBEDDINGS = True

# Keep this True when every output channel must have all three source descriptions.
# Set to False to include the union of channels and allow missing source keys.
REQUIRE_ALL_SOURCES = True

print('Source artifacts:')
for source, path in SOURCE_ARTIFACTS.items():
    print(f'  {source}: {path}')
print(f'Output path: {OUTPUT_PATH}')
print(f'Similarity matrix output directory: {SIMILARITY_OUTPUT_DIR}')
print(f'Embedding output directory: {EMBEDDING_OUTPUT_DIR}')
print(f'Embedding model: {EMBEDDING_MODEL_NAME}')


## 2) Load and normalize source artifacts

The three notebooks write different artifact shapes: `Jules` writes one channel-to-object JSON file, `codex` writes one JSON file per channel, and `copilot` writes a combined `channels` object. These helpers normalize all three forms into `channel_name -> artifact` mappings and extract the richest text description available.


In [ ]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def as_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, str):
        return value.strip()
    return json.dumps(value, ensure_ascii=False, sort_keys=True).strip()


def extract_description(payload: dict[str, Any]) -> str:
    parts: list[str] = []

    # Jules and copilot use `description`; codex uses short/detailed fields.
    for key in ('description', 'short_description', 'detailed_description'):
        text = as_text(payload.get(key))
        if text:
            parts.append(text)

    if parts:
        return '\n\n'.join(dict.fromkeys(parts))

    # Fallback for any future structured description artifact.
    return as_text(payload)


def load_channel_mapping(path: Path) -> dict[str, dict[str, Any]]:
    if path.is_dir():
        channels: dict[str, dict[str, Any]] = {}
        for filename in sorted(glob.glob(str(path / '*.json'))):
            payload = read_json(Path(filename))
            if not isinstance(payload, dict):
                continue
            channel = payload.get('channel_name') or payload.get('channel') or Path(filename).stem
            channels[str(channel)] = payload
        return channels

    payload = read_json(path)
    if not isinstance(payload, dict):
        raise ValueError(f'Expected a JSON object at {path}, got {type(payload).__name__}')

    if isinstance(payload.get('channels'), dict):
        return {
            str(channel): artifact if isinstance(artifact, dict) else {'description': artifact}
            for channel, artifact in payload['channels'].items()
        }

    return {
        str(channel): artifact if isinstance(artifact, dict) else {'description': artifact}
        for channel, artifact in payload.items()
    }


## 3) Merge descriptions

Build the channel-indexed object. With `REQUIRE_ALL_SOURCES = True`, only channels present in all three source artifacts are included, so each channel has exactly three descriptions.


In [ ]:
def merge_descriptions(
    source_artifacts: dict[str, Path],
    *,
    require_all_sources: bool = True,
) -> dict[str, Any]:
    loaded = {source: load_channel_mapping(path) for source, path in source_artifacts.items()}
    source_sets = {source: set(channels) for source, channels in loaded.items()}
    all_channels = set.union(*source_sets.values()) if source_sets else set()

    if require_all_sources:
        selected_channels = set.intersection(*source_sets.values()) if source_sets else set()
    else:
        selected_channels = all_channels

    merged_channels: dict[str, dict[str, str]] = {}
    for channel in sorted(selected_channels):
        descriptions_by_source: dict[str, str] = {}
        for source in sorted(source_artifacts):
            artifact = loaded[source].get(channel)
            if artifact is None:
                continue
            description = extract_description(artifact)
            if description:
                descriptions_by_source[source] = f'{source}: {description}'
        if (not require_all_sources) or len(descriptions_by_source) == len(source_artifacts):
            merged_channels[channel] = descriptions_by_source

    return {
        'schema_name': 'graphiko.research.merged_channel_descriptions_by_source',
        'schema_version': '1.0.0',
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'description': (
            'Channel-indexed descriptions merged from the Jules, codex, and copilot '
            'research notebook Drive artifacts. Each description value is prefixed '
            'by its source subdirectory name.'
        ),
        'require_all_sources': require_all_sources,
        'sources': {source: str(path) for source, path in source_artifacts.items()},
        'source_channel_counts': {source: len(channels) for source, channels in loaded.items()},
        'channel_count': len(merged_channels),
        'omitted_channels_by_source': {
            source: sorted(all_channels - channels)
            for source, channels in source_sets.items()
        },
        'channels': merged_channels,
    }


## 4) Write the merged artifact

Validate that the Drive artifacts exist, then write the merged object to the configured `OUTPUT_PATH`. The default output location is documented in `research/README.md`.


In [ ]:
missing_sources = {source: path for source, path in SOURCE_ARTIFACTS.items() if not path.exists()}
if missing_sources:
    print('Missing source artifact(s). Update SOURCE_ARTIFACTS above or run the source notebooks first:')
    for source, path in missing_sources.items():
        print(f'  {source}: {path}')
else:
    merged = merge_descriptions(SOURCE_ARTIFACTS, require_all_sources=REQUIRE_ALL_SOURCES)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open('w', encoding='utf-8') as fh:
        json.dump(merged, fh, ensure_ascii=False, indent=2)
        fh.write('\n')

    print(f"Wrote {merged['channel_count']} merged channel records to {OUTPUT_PATH}")
    print('Source channel counts:')
    for source, count in merged['source_channel_counts'].items():
        print(f'  {source}: {count}')


## 5) Encode descriptions and write cosine-similarity matrices

For each source (`Jules`, `codex`, and `copilot`), encode that source's channel descriptions with the configured Sentence Transformers model. Because embeddings are normalized during encoding, the cosine similarity between two channels is the dot product of their vectors.

The notebook writes one symmetric CSV matrix per source. Rows and columns are channels in the same order, and the diagonal is the self-similarity value for each channel. It can also write per-source embedding vectors for reproducibility.


In [ ]:
from sentence_transformers import SentenceTransformer


def safe_filename(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('_') or 'source'


def text_for_embedding(source: str, description: str, *, include_source_prefix: bool) -> str:
    if include_source_prefix:
        return description

    prefix = f'{source}: '
    if description.startswith(prefix):
        return description[len(prefix):].strip()
    return description.strip()


def cosine_similarity_matrix_from_normalized_embeddings(
    embeddings: list[list[float]],
) -> list[list[float]]:
    matrix = [[0.0 for _ in embeddings] for _ in embeddings]
    for i, left in enumerate(embeddings):
        for j in range(i, len(embeddings)):
            right = embeddings[j]
            similarity = float(sum(a * b for a, b in zip(left, right)))
            matrix[i][j] = similarity
            matrix[j][i] = similarity
    return matrix


def write_similarity_matrix_csv(
    path: Path,
    channels: list[str],
    matrix: list[list[float]],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as fh:
        writer = csv.writer(fh)
        writer.writerow(['channel', *channels])
        for channel, row in zip(channels, matrix):
            writer.writerow([channel, *[f'{value:.10f}' for value in row]])


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)
        fh.write('\n')


def build_description_similarity_artifacts(
    merged: dict[str, Any],
    *,
    matrix_output_dir: Path,
    embedding_output_dir: Path,
    model_name: str,
    include_source_prefix: bool = False,
    write_embeddings: bool = True,
) -> dict[str, Any]:
    channels_payload = merged.get('channels', {})
    sources = sorted(merged.get('sources', SOURCE_ARTIFACTS).keys())
    model = SentenceTransformer(model_name)
    artifacts: dict[str, Any] = {}

    for source in sources:
        channel_names: list[str] = []
        descriptions: list[str] = []
        for channel, descriptions_by_source in sorted(channels_payload.items()):
            description = descriptions_by_source.get(source, '')
            encoded_text = text_for_embedding(
                source,
                description,
                include_source_prefix=include_source_prefix,
            )
            if encoded_text:
                channel_names.append(channel)
                descriptions.append(encoded_text)

        if not descriptions:
            print(f'Skipping {source}: no descriptions found in merged artifact.')
            continue

        embeddings = model.encode(
            descriptions,
            normalize_embeddings=True,
            show_progress_bar=True,
        ).tolist()
        matrix = cosine_similarity_matrix_from_normalized_embeddings(embeddings)
        source_slug = safe_filename(source)
        matrix_path = matrix_output_dir / f'{source_slug}_description_cosine_similarity.csv'
        write_similarity_matrix_csv(matrix_path, channel_names, matrix)

        embedding_path = None
        if write_embeddings:
            embedding_path = embedding_output_dir / f'{source_slug}_description_embeddings.json'
            write_json(
                embedding_path,
                {
                    'schema_name': 'graphiko.research.description_embeddings_by_source',
                    'schema_version': '1.0.0',
                    'source': source,
                    'model_name': model_name,
                    'include_source_prefix': include_source_prefix,
                    'channel_count': len(channel_names),
                    'channels': channel_names,
                    'embeddings': embeddings,
                },
            )

        artifacts[source] = {
            'channel_count': len(channel_names),
            'matrix_path': str(matrix_path),
            'embedding_path': str(embedding_path) if embedding_path else None,
        }
        print(f'Wrote {source} cosine-similarity matrix: {matrix_path}')

    manifest = {
        'schema_name': 'graphiko.research.description_cosine_similarity_matrices',
        'schema_version': '1.0.0',
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'model_name': model_name,
        'include_source_prefix': include_source_prefix,
        'matrix_output_dir': str(matrix_output_dir),
        'embedding_output_dir': str(embedding_output_dir),
        'sources': artifacts,
    }
    write_json(matrix_output_dir / 'manifest.json', manifest)
    return manifest


if OUTPUT_PATH.exists():
    merged_for_similarity = read_json(OUTPUT_PATH)
    similarity_manifest = build_description_similarity_artifacts(
        merged_for_similarity,
        matrix_output_dir=SIMILARITY_OUTPUT_DIR,
        embedding_output_dir=EMBEDDING_OUTPUT_DIR,
        model_name=EMBEDDING_MODEL_NAME,
        include_source_prefix=INCLUDE_SOURCE_PREFIX_IN_EMBEDDINGS,
        write_embeddings=WRITE_DESCRIPTION_EMBEDDINGS,
    )
    print(json.dumps(similarity_manifest, indent=2))
else:
    print(f'No merged artifact exists yet; skipping similarity matrices: {OUTPUT_PATH}')


## 6) Preview the output

Load a small preview from the just-written artifact to confirm that each channel contains source-prefixed descriptions.


In [ ]:
if OUTPUT_PATH.exists():
    preview = read_json(OUTPUT_PATH)
    print(json.dumps({
        'schema_name': preview.get('schema_name'),
        'channel_count': preview.get('channel_count'),
        'first_channel': next(iter(preview.get('channels', {})), None),
        'first_channel_sources': (
            list(next(iter(preview.get('channels', {}).values())).keys())
            if preview.get('channels')
            else []
        ),
    }, indent=2))
else:
    print(f'No output artifact exists yet: {OUTPUT_PATH}')
